# GAN: 生成器と識別器の交互最適化

GANは、偽物を作るGeneratorと本物か偽物かを見分けるDiscriminatorを競わせるモデルである。


## このノートの読み方

想定読者: 二値分類、BCE、簡単なPyTorchのoptimizerを理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

MLPの二値分類器をDiscriminatorとして使う。ただし通常の教師あり学習と違い、GeneratorとDiscriminatorの目的が同時に変化する。


## 到達目標

- D更新とG更新を分けて説明できる
- `fake.detach()`の意味を説明できる
- mode collapseを失敗例として説明できる


## 重要語句

- `Generator`: zからfake sampleを作るモデル
- `Discriminator`: real/fakeを分類するモデル
- `mode collapse`: 多様性が失われる失敗


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| z | (B, z_dim) | 潜在ノイズ |
| G(z) | (B, data_dim) | 偽物データ |
| D(x) | (B, 1) | 本物らしさlogit |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| minimaxと実装loss | 理論式はminimaxだが、実装ではGeneratorに`BCE(D(G(z)),1)`を使うnon-saturating lossがよく使われる。勾配が消えにくいからである。 |
| Trainerの位置づけ | GANはDとGでoptimizerを分けるため、標準Trainerの単一loss学習とは相性が悪い。このノートでは`Trainer`を継承した交互更新デモとして扱う。 |
| detach | D更新時の`fake.detach()`は、偽物データを使ってDだけを更新し、Gへ勾配を流さないための境界である。 |
| 多様性 | 細胞画像生成なら、見た目が鮮明でも細胞状態の多様性を失うと危険である。mode collapseは品質と別に確認する。 |


## Minimax objective

理論上は二者ゲームとして定義される。

$$
\min_G\max_D\ \mathbb{E}_{x\sim p_{\mathrm{data}}}\log D(x)+\mathbb{E}_{z}\log(1-D(G(z)))
$$


## D update

D更新ではfakeをdetachし、Gへ勾配を流さない。

$$
L_D=\mathrm{BCE}(D(x),1)+\mathrm{BCE}(D(G(z)),0)
$$


## G update

G更新ではDを固定し、Dをだます方向へGだけ更新する。

$$
L_G=\mathrm{BCE}(D(G(z)),1)
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
G = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
D = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
bce = nn.BCEWithLogitsLoss()
real = torch.randn(4, 2) + 1.0
z = torch.randn(4, 2)
fake = G(z)
d_loss = bce(D(real), torch.ones(4, 1)) + bce(D(fake.detach()), torch.zeros(4, 1))
for p in D.parameters():
    p.requires_grad_(False)
g_loss = bce(D(fake), torch.ones(4, 1))
for p in D.parameters():
    p.requires_grad_(True)
print("D loss:", float(d_loss.detach()), "G loss:", float(g_loss.detach()))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/gan_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/gan_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/gan_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="GAN: 生成器と識別器の交互最適化 difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### G/D alternating update animation

- 学習目標: D更新とG更新を分けて見せる
- 誤解の防止: 1つのlossで同時更新すればよいと思う

対応する式:

$$
D\ step;\ G\ step
$$


<p><a href="../demos/gan_alternating.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/gan_alternating.html</code>）</p>
<iframe
  src="../demos/gan_alternating.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="G/D alternating update animation"
></iframe>


### fake detach animation

- 学習目標: D更新でGへ勾配を流さない
- 誤解の防止: fake経由でGも更新されると思う

対応する式:

$$
D(G(z).detach())
$$


<p><a href="../demos/gan_fake_detach.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/gan_fake_detach.html</code>）</p>
<iframe
  src="../demos/gan_fake_detach.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="fake detach animation"
></iframe>


### discriminator boundary animation

- 学習目標: 判定境界が変わる様子を見せる
- 誤解の防止: Dは固定だと思う

対応する式:

$$
D_\phi(x)
$$


<p><a href="../demos/gan_boundary.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/gan_boundary.html</code>）</p>
<iframe
  src="../demos/gan_boundary.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="discriminator boundary animation"
></iframe>


### mode collapse animation

- 学習目標: 多くのzが同じ出力へ潰れる
- 誤解の防止: 品質だけ見ればよいと思う

対応する式:

$$
z_1,z_2,\ldots \mapsto G(z)\approx x_a
$$


<p><a href="../demos/gan_collapse.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/gan_collapse.html</code>）</p>
<iframe
  src="../demos/gan_collapse.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="mode collapse animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`GAN: 生成器と識別器の交互最適化`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyGANDataset(Dataset):
    def __init__(self, n_samples: int = 40, z_dim: int = 2) -> None:
        self.real = torch.randn(n_samples, 2) * 0.3 + torch.tensor([1.0, 1.0])
        self.noise = torch.randn(n_samples, z_dim)

    def __len__(self) -> int:
        return len(self.real)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"real": self.real[index], "noise": self.noise[index]}


class TinyGANBundle(nn.Module):
    def __init__(self, z_dim: int = 2) -> None:
        super().__init__()
        self.generator = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, 2))
        self.discriminator = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))


class TinyGANTrainer(Trainer):
    # GANはD/Gでoptimizerを分けるため、標準Trainerの単一lossではなく交互更新を明示する。
    def train(self, *args, **kwargs):
        from types import SimpleNamespace

        g_optimizer = torch.optim.Adam(self.model.generator.parameters(), lr=self.args.learning_rate)
        d_optimizer = torch.optim.Adam(self.model.discriminator.parameters(), lr=self.args.learning_rate)
        bce = nn.BCEWithLogitsLoss()
        self.model.train()
        metrics = {"d_loss": 0.0, "g_loss": 0.0}
        for step, batch in enumerate(self.get_train_dataloader()):
            if step >= int(self.args.max_steps):
                break
            batch = self._prepare_inputs(batch)
            real = batch["real"]
            noise = batch["noise"]

            d_optimizer.zero_grad()
            fake_for_d = self.model.generator(noise).detach()
            d_real = self.model.discriminator(real)
            d_fake = self.model.discriminator(fake_for_d)
            d_loss = bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))
            d_loss.backward()
            d_optimizer.step()

            g_optimizer.zero_grad()
            for parameter in self.model.discriminator.parameters():
                parameter.requires_grad_(False)
            fake_for_g = self.model.generator(noise)
            d_fake_for_g = self.model.discriminator(fake_for_g)
            g_loss = bce(d_fake_for_g, torch.ones_like(d_fake_for_g))
            g_loss.backward()
            g_optimizer.step()
            for parameter in self.model.discriminator.parameters():
                parameter.requires_grad_(True)
            metrics = {"d_loss": float(d_loss.detach()), "g_loss": float(g_loss.detach())}

        return SimpleNamespace(training_loss=metrics["d_loss"] + metrics["g_loss"], metrics=metrics)


training_args = TrainingArguments(
    output_dir="./results/gan_trainer_demo",
    max_steps=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    remove_unused_columns=False,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = TinyGANTrainer(model=TinyGANBundle(), args=training_args, train_dataset=TinyGANDataset())
train_output = trainer.train()
with torch.no_grad():
    z_many = torch.randn(8, 2)
    fake_many = trainer.model.generator(z_many)
    diversity = fake_many.std(dim=0).mean()
print("GAN custom Trainer demo loss:", train_output.training_loss)
print("fake diversity:", float(diversity.detach()))


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- D/G lossを別々にプロットする
- mode collapseの分布を可視化する
- WGANやDCGANへ進む


## 確認問題

- D更新時に`fake.detach()`が必要な理由を書く。
- mode collapseの兆候を1つ挙げる。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
